## A durable mental model

Python code normally works with **names**, **objects**, and **references**:

```text
name in a namespace  ──references──>  object (value + type + state)
```

- A name is an entry in a namespace: a function's local namespace, a module's global namespace, an object's attributes, or a container such as a list or dictionary.
- Assignment (`target = value`) evaluates the value and binds the target to the resulting object. It does **not** copy that object.
- Multiple names or containers can refer to one mutable object. A mutation is therefore visible through every alias. Rebinding one name changes only that one binding.
- Function calls follow the same rule: parameters become new local names referring to the caller's objects. Python is often described as *call by object reference* or *call by sharing*.
- An object's lifetime is about reachability/references, not the spelling of any particular variable. `del name` removes a binding; it does not directly mean “free this object.”

Keep the language-level model separate from implementation details. `id()` is a stable identity only while the object lives; on CPython it is commonly related to the address, but portable Python code must not treat it as an address. Likewise, reference counts, integer caching, and exact collection timing are implementation details.

**Retention prompt:** Before running each later cell, say: (1) which objects exist, (2) which references point to each object, (3) whether the next operation mutates an object or rebinds a name, and (4) which objects can become unreachable.

## Stack vs. heap in Python (memory-management meaning)

This distinction is a useful approximation, but **it is not a Python language guarantee**. The interpreter decides its actual memory layout. Use it to reason about lifetimes and calls—not to infer object addresses or optimize ordinary code.

| Idea | Helpful Python interpretation | Important correction |
| --- | --- | --- |
| **Call stack** | Active function calls are nested. Each call has an execution frame containing that call's local namespace, instruction state, and bookkeeping. Returning normally makes that frame cease to be active. | A Python frame is not simply a C stack slot. Implementations can allocate, optimize, retain, or expose frames differently. A traceback, generator, coroutine, closure, or `sys._getframe()` can keep frame-related state alive beyond a normal return. |
| **Heap** | Python objects—such as lists, dictionaries, class instances, and usually integers/strings—are dynamically allocated objects whose lifetime can outlast the call that created them. Containers and locals hold references to them. | “Heap object” does **not** mean “global” or “slow.” An object created in a function can safely be returned because another reference now keeps it reachable. |

For a typical CPython intuition, draw a vertical stack of active calls and draw objects separately. Inside each frame, local names point outward to objects. When `build_report()` returns a list, its local binding disappears, but the caller's binding points to the same list, so the list remains alive. When the last relevant reference disappears, CPython usually destroys a non-cyclic object immediately through reference counting; its cyclic garbage collector handles unreachable cycles later. Other Python implementations may use different collection strategies.

A common misconception is “local variables live on the stack and objects live on the heap.” A better statement is: **a local name belongs to a call's frame; the object it refers to has its own lifetime.** The local name is a reference, not an inline copy of every list or instance it names.

**Retention prompt:** If a function returns a list, which thing disappears at return time—the local name, the list object, or both? Answer: the local binding disappears; the list survives if the returned reference is stored elsewhere.

In [ ]:
# CELL 1.5 — Call frames hold names; returned objects can outlive the call

def build_artifact_paths() -> list[str]:
    paths = ["gs://bucket/run-17/model"]  # local name in this call's frame
    print("inside function, paths id:", id(paths))
    return paths                        # return one reference to the caller

artifact_paths = build_artifact_paths()
print("after return, paths id:", id(artifact_paths))
assert artifact_paths == ["gs://bucket/run-17/model"]
# The name `paths` is no longer a local binding in an active call,
# but the list remains reachable through `artifact_paths`.


## Object lifetime, cleanup, and what `del` means

There are two separate questions:

1. **Can the program still reach the object?** References from names, containers, instance attributes, closures, running tasks, and sometimes tracebacks may keep it alive.
2. **When does the implementation reclaim its storage?** That timing is implementation-specific. CPython combines reference counting with a cyclic garbage collector; do not use collection timing as application control flow.

Use deterministic resource-management protocols for things that need prompt external cleanup—files, sockets, database cursors, locks, and cloud responses—rather than waiting for memory collection. `with open(...) as f:` closes the file even if an exception occurs. The file object is Python memory; the open file descriptor is an external resource, and they have different lifetimes.

The later `sys.getrefcount` cells are deliberately labelled CPython-specific. They are useful for experiments, but avoid production logic such as “close a client when its reference count is one.”

In [ ]:
# CELL 1.6 — `del` removes a binding; an alias can keep the object alive

run_metadata = {"run_id": "r-42"}
alias = run_metadata
original_id = id(run_metadata)

del run_metadata  # removes this name, not necessarily the dictionary
assert alias["run_id"] == "r-42"
assert id(alias) == original_id

# After `del alias`, this example leaves no user-created name pointing to the dict.
# Do not try to observe its destruction with application logic: reclamation is
# interpreter-specific, and CPython may reuse a later id.
del alias


In [ ]:
# ============================================================
# Python Memory Management & Object Model — Mini Example Notebook
# Each section below is intended to be its own Jupyter cell.
# Copy/paste cell-by-cell.
# ============================================================


In [ ]:
# CELL 1 — Helper utilities for observing identity and aliases
# Purpose: make later experiments inspectable without changing their semantics.
# `id()` helps compare identity, `is` compares identity directly, and the
# refcount helper is only an optional CPython observation—not portable logic.

import copy
import sys
import gc

def show(name, obj):
    """Print name, repr, type, id."""
    print(f"{name}: {obj!r} | type={type(obj).__name__} | id={id(obj)}")

def same(a, b, label="same object?"):
    print(f"{label}: {a is b}")

def refcount(obj):
    # sys.getrefcount adds 1 because passing obj as an argument creates a temporary reference
    return sys.getrefcount(obj) - 1


In [ ]:
# CELL 2 — Names bind to objects
# Predict: after x is rebound, will y change? No: assignment changes the
# binding of x; it does not mutate or copy the object previously named x.

x = 1
y = x
show("x", x); show("y", y)
assert y == 1
assert x is y  # often True for small ints due to caching, but still ok here

x = 2
show("x", x); show("y", y)
assert x == 2
assert y == 1
assert x is not y


In [ ]:
# CELL 3 — Mutation is visible through every alias
# Predict: x and y initially name one list. `append` changes that list in
# place, so both names observe the new contents and keep the same identity.

x = [1]
y = x
show("x", x); show("y", y)
assert x is y

x.append(2)  # mutation
show("x", x); show("y", y)
assert x == [1, 2]
assert y == [1, 2]
assert x is y


In [ ]:
# CELL 4 — Rebinding does not affect other names
# Contrast with Cell 3: this assignment makes x name a *new* list. The old
# list is still reachable through y, so y keeps seeing its original value.

x = [1, 2]
y = x
x = [1, 2, 3]  # rebinding
show("x", x); show("y", y)
assert x == [1, 2, 3]
assert y == [1, 2]
assert x is not y


In [ ]:
# CELL 5 — Immutable values are replaced, not changed in place
# Integers cannot be mutated. `x += 1` computes another integer and rebinds
# x; y still refers to the earlier integer. Do not infer anything from ids
# beyond this experiment—identity reuse/caching is implementation-specific.

x = 10
y = x
show("x", x); show("y", y)

x += 1  # creates a new int object and rebinds x
show("x", x); show("y", y)
assert x == 11
assert y == 10
assert x is not y


In [ ]:
# CELL 6 — Strings follow the same immutable/rebinding rule
# Predict: concatenation makes a new string value; it cannot edit the string
# named by y. This is why repeated string concatenation can allocate work.

x = "hi"
y = x
show("x", x); show("y", y)

x += "!"
show("x", x); show("y", y)
assert x == "hi!"
assert y == "hi"
assert x is not y


In [ ]:
# CELL 7 — An immutable container can contain mutable objects
# A tuple fixes its own sequence of references, not the state of objects those
# references target. Here the tuple still points at the same list after append.

t = ([1, 2], 99)
show("t", t)
t[0].append(3)  # mutates the list inside the tuple
show("t after", t)

assert t == ([1, 2, 3], 99)
# Note: tuple identity doesn't change, but contents observed via refs do


In [ ]:
# CELL 8 — Assignment creates an alias, not a copy
# Predict: b is not an independent nested list. Since a and b are two names
# for one outer list, mutating through a is immediately observable through b.

a = [[1, 2], [3, 4]]
b = a
same(a, b, "a is b")
assert a is b

a[0].append(99)
print("a:", a)
print("b:", b)
assert b[0] == [1, 2, 99]


In [ ]:
# CELL 9 — Shallow versus deep copy
# A shallow copy duplicates only the outer container and retains references to
# its children. deepcopy recursively copies the reachable object graph (with
# memoization), so use it deliberately: it can be costly or semantically wrong.

import copy

a = [[1, 2], [3, 4]]
b = copy.copy(a)       # shallow: new outer list, shared inner lists
c = copy.deepcopy(a)   # deep: fully independent

show("a", a); show("b", b); show("c", c)
assert a is not b
assert a is not c

# Inner lists:
same(a[0], b[0], "a[0] is b[0] (shallow shares inner)")
same(a[0], c[0], "a[0] is c[0] (deep does not share inner)")
assert a[0] is b[0]
assert a[0] is not c[0]

a[0].append(99)
print("a:", a)
print("b:", b)
print("c:", c)

assert b[0] == [1, 2, 99]   # shared inner list
assert c[0] == [1, 2]       # independent copy


In [ ]:
# CELL 10 — List slicing is a shallow copy
# `a[:]` allocates a different outer list, then copies references into it.
# Predict the inner-list mutation before running: both outer lists share it.

a = [[1, 2], [3, 4]]
b = a[:]  # shallow copy

assert a is not b
assert a[0] is b[0]  # inner list shared

a[0].append(99)
print("a:", a)
print("b:", b)
assert b[0] == [1, 2, 99]


In [ ]:
# CELL 11 — dict.copy() is shallow too
# The dictionary table is new, but its values are the same referenced objects.
# Copy structure only when sharing nested mutable values is intentional/safe.

a = {"k": [1, 2]}
b = a.copy()

assert a is not b
assert a["k"] is b["k"]  # same list value shared

a["k"].append(3)
print("a:", a)
print("b:", b)
assert b["k"] == [1, 2, 3]


In [ ]:
# CELL 12 — Copies of immutable values may reuse the original object
# Because this tuple and all of its contents are immutable, a copy operation
# may safely return the original. Treat this as an optimization, not a promise:
# correctness should rely on value/immutability, never `copy(x) is x`.

a = (1, 2, 3)
b = copy.copy(a)
c = copy.deepcopy(a)

# For immutables, copy/deepcopy can return the original object safely
print("a is b:", a is b)
print("a is c:", a is c)
assert a == b == c == (1, 2, 3)


In [ ]:
# CELL 13 — Function arguments bind another name to the caller's object
# Python does not automatically copy arguments. The parameter `lst` aliases a,
# so an in-place append affects the caller. Rebinding lst would not rebind a.

def f(lst):
    lst.append("X")

a = []
f(a)
print(a)
assert a == ["X"]


In [ ]:
# CELL 14 — GOTCHA: default expressions are evaluated once, at function definition
# The bad function's one list is then reused on every call. Use None as a
# sentinel and allocate inside the function when each call needs fresh state.

def bad_append(x, bucket=[]):
    bucket.append(x)
    return bucket

r1 = bad_append(1)
r2 = bad_append(2)
print("r1:", r1)
print("r2:", r2)
assert r1 is r2
assert r1 == [1, 2]

# Fix:
def good_append(x, bucket=None):
    if bucket is None:
        bucket = []
    bucket.append(x)
    return bucket

r1 = good_append(1)
r2 = good_append(2)
print("good r1:", r1)
print("good r2:", r2)
assert r1 == [1]
assert r2 == [2]
assert r1 is not r2


In [ ]:
# CELL 15 — `==` asks about value; `is` asks about the very same object
# Equal lists need not be identical. Prefer `==` for domain values and reserve
# `is` for singleton identity checks such as `value is None`.

a = [1, 2]
b = [1, 2]

print("a == b:", a == b)  # value equality
print("a is b:", a is b)  # identity
assert a == b
assert a is not b

# When does "is" make sense? Mostly with singletons like None.
x = None
assert x is None


In [ ]:
# CELL 16 — `+=` may mutate; `+` produces a new list
# For lists, `a += ...` uses in-place extension, so aliases see the change.
# `a = a + ...` builds another list and then only rebinds a. Check a type's
# documented behavior rather than assuming every `+=` has the same semantics.

a = [1, 2]
b = a
a += [3]  # in-place extend (mutates list), then rebinds a to same object
print("a:", a, "id:", id(a))
print("b:", b, "id:", id(b))
assert a is b
assert b == [1, 2, 3]

# Compare with + (creates a new list):
a = [1, 2]
b = a
a = a + [3]  # new list object
print("a:", a, "id:", id(a))
print("b:", b, "id:", id(b))
assert a is not b
assert b == [1, 2]


In [ ]:
# CELL 17 — `+=` on an immutable integer requires a replacement value
# This contrasts with Cell 16: integers have no in-place state to extend, so
# a changes binding and b continues to name the original integer.

a = 10
b = a
a += 5
show("a", a); show("b", b)
assert a == 15
assert b == 10
assert a is not b


In [ ]:
# CELL 18 — Small-integer identity is a CPython implementation detail
# This intentionally demonstrates an observation, not a rule. Literal handling
# and caching can make `is` appear useful for integers; always use `==` instead.

# CPython usually interns/caches small integers (commonly -5..256).
# This is NOT a language guarantee, but is common.
a = 256
b = 256
print("256: a is b ->", a is b)

a = 257
b = 257
print("257: a is b ->", a is b)

# Don't write code that relies on this:
assert (256 == 256)
assert (257 == 257)


In [ ]:
# CELL 19 — Identity observations can change with code shape and interpreter
# Moving literals into functions may alter compiler/interpreter behavior. The
# stable lesson is that `==` is semantic equality; identity is not a number test.

def small_int_demo():
    x = 256
    y = 256
    return x is y, x == y

def bigger_int_demo():
    x = 10_000
    y = 10_000
    return x is y, x == y

print("small_int_demo (is, ==):", small_int_demo())
print("bigger_int_demo (is, ==):", bigger_int_demo())


In [ ]:
# CELL 20 — String interning is also an implementation detail
# Equal strings may or may not share storage. `sys.intern` can be useful for
# specialized workloads, but it does not turn `is` into a general string check.

# Some strings (identifiers, literals) may be interned.
a = "hello_world"
b = "hello_world"
print("literal string: a is b ->", a is b)

# Building strings at runtime may not be interned:
x = "".join(["hello", "_", "world"])
print("runtime built: a is x ->", a is x)
print("a == x ->", a == x)
assert a == x

# You can force interning:
import sys
i1 = sys.intern(x)
print("after intern: a is i1 ->", a is i1)
assert a == i1


In [ ]:
# CELL 21 — Do not use `is` to compare ordinary string values
# Constant folding or interning can make the identity result vary. Predict only
# the guaranteed result: the two strings have equal content.

a = "py" * 3
b = "pypypy"
print("a == b:", a == b)
print("a is b:", a is b)  # could be True or False depending on interning
assert a == b


In [ ]:
# CELL 22 — Reference-counting intuition (CPython-specific diagnostic)
# Adding b adds a reference to the same list; deleting b removes that binding.
# `getrefcount` itself creates a temporary reference, hence the helper adjustment.
# Never make program behavior depend on an exact count: it is noisy and nonportable.

# Warning: refcount observations are CPython-specific and can be noisy.

a = []
show("a", a)
print("refcount(a) before alias:", refcount(a))

b = a
print("refcount(a) after alias b=a:", refcount(a))

del b
print("refcount(a) after del b:", refcount(a))

# a is still alive because name 'a' still refers to it
assert isinstance(a, list)


In [ ]:
# CELL 23 — Reference cycles need cyclic garbage collection on CPython
# The list refers to itself, so its internal reference survives after its external
# name is deleted. Re-enabling GC and collecting demonstrates reclamation.
# Avoid relying on this timing; use context managers for prompt external cleanup.

gc.disable()

a = []
a.append(a)  # a contains itself => cycle
show("a", a)
print("refcount(a):", refcount(a))

# Remove the last external reference:
a_id = id(a)
del a

# At this point, the cyclic object may still exist until GC runs.
# Force a collection:
gc.enable()
unreachable = gc.collect()
print("gc.collect() reclaimed objects:", unreachable)
print("cycle demo done (cannot directly assert object freed)")


In [ ]:
# CELL 24 — Predict-before-run drills: outer copies versus nested aliases
# Drill A: a slice separates the outer list, so appending to a changes only a.
# Drill B: the slice still shares the inner lists, so their mutation appears in b.
# Draw the references before executing; prediction builds the durable intuition.

# Drill A
a = [1, 2, 3]
b = a
a = a[:]       # shallow copy (new list)
a.append(4)
print("a:", a)
print("b:", b)
assert a == [1, 2, 3, 4]
assert b == [1, 2, 3]

# Drill B
a = [[0], [1]]
b = a[:]       # shallow copy
a[0].append(9) # mutate shared inner list
print("a:", a)
print("b:", b)
assert b[0] == [0, 9]


In [ ]:
# CELL 25 — Final comparison: mutation versus rebinding
# First `append` changes the one shared list, keeping x is y true. Then the
# assignment points x at another list, leaving y attached to the old one.
# This is the core diagram to recall when debugging surprising shared state.

x = [1, 2]
y = x
show("x", x); show("y", y)

# mutation
x.append(3)
print("After mutation x.append(3):")
show("x", x); show("y", y)
assert x is y

# rebinding
x = [7, 8]
print("After rebinding x = [7, 8]:")
show("x", x); show("y", y)
assert x is not y


In [ ]:
# CELL 26 — Reusable predict-before-run practice harness
# Use this only with trusted, small snippets: exec runs arbitrary Python code.
# For each snippet, write object/reference predictions before revealing output.

def predict_then_run(code_str, *, globals_dict=None):
    print("PREDICT FIRST — then run:\n")
    print(code_str)
    print("\n--- output ---")
    exec(code_str, globals_dict if globals_dict is not None else {})

predict_then_run("""
a = [[1],[2]]
b = a.copy()
a[0].append(99)
print(a)
print(b)
""")


## Before you leave: five-question memory-management checklist

When a Python value surprises you, pause before reaching for `copy`, `gc`, or `del` and ask:

1. What are the distinct objects, and which names/containers refer to each one?
2. Did the operation mutate an object or rebind one name?
3. If a copy occurred, was it shallow or deep—and which nested objects remain shared?
4. Is this a language guarantee, or merely a CPython observation (identity caching, refcounts, GC timing)?
5. Am I managing Python memory, or an external resource that needs a `with` block / explicit `close()`?

If you can answer these, you can usually predict the output, explain the lifetime, and choose the simplest correct fix.